In [43]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

import matplotlib.pyplot as plt

In [44]:
df = pd.read_csv("AAPL.csv")
close = np.array(df["Close"])
returns = close[1:]/close[:-1]
log_returns = np.log(returns)

rvolwin  = 10
windows = sliding_window_view(log_returns, window_shape=rvolwin)
rvol = windows.std(axis=1, ddof=1)

log_returns = log_returns[rvolwin-1:]

combined = np.vstack((rvol, 
              log_returns))

rvol_sd = np.std(rvol)
rvol_mean = np.mean(rvol)

log_returns_sd = np.std(log_returns)
log_returns_mean = np.mean(log_returns)

combined[0,:] =  (combined[0,:] - rvol_mean)/rvol_sd
combined[1,:] =  (combined[1,:] - log_returns_mean)/log_returns_sd

seq_len = 50

X = []
Y = []

for i in range(combined.shape[1] - seq_len):
    X.append(combined[:,i:(seq_len + i)])
    Y.append(log_returns[i + seq_len] > 0)

X = torch.tensor(X).float()
Y = torch.tensor(Y).float()

X = X.reshape((X.shape[0],
               X.shape[2],
               X.shape[1]))

test_size = 1000

test_X = X[X.shape[0]- test_size:, :, :]
test_Y = Y[Y.shape[0] - test_size:]


train_X = X[:-test_size, :, :]
train_Y = Y[:-test_size]


In [45]:
class model(nn.Module):
    def __init__(self, input_size=2, hidden_size=100, output_size=1):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = x.transpose(0, 1)                    
        out, _ = self.rnn(x)
        out = self.fc(out[-1])                  
        return out                            


train_dataset = TensorDataset(train_X, train_Y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(test_X, test_Y)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model = model()
criterion = nn.BCEWithLogitsLoss()              
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x).squeeze(-1)      
        loss = criterion(outputs, batch_y)        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x).squeeze(-1)    
        loss = criterion(outputs, batch_y)
        test_loss += loss.item() * batch_x.size(0)
        
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

test_loss /= len(test_loader.dataset)
accuracy = correct / total if total > 0 else 0

print(f"Test BCE Loss: {test_loss:.4f}")
print(f"Test: {accuracy:.4f}  ({correct}/{total})")
print(f"All up: {test_Y.mean():.4f}")

Epoch 1/10, Loss: 0.6845
Epoch 2/10, Loss: 0.6824
Epoch 3/10, Loss: 0.6827
Epoch 4/10, Loss: 0.6831
Epoch 5/10, Loss: 0.6832
Epoch 6/10, Loss: 0.6833
Epoch 7/10, Loss: 0.6831
Epoch 8/10, Loss: 0.6828
Epoch 9/10, Loss: 0.6823
Epoch 10/10, Loss: 0.6816
Test BCE Loss: 0.6915
Test: 0.5100  (510/1000)
All up: 0.5260


In [39]:
class model(nn.Module):
    def __init__(self, input_size=2, hidden_size=100, output_size=1):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = x.transpose(0, 1)                    
        out, _ = self.rnn(x)
        out = self.fc(out[-1])                  
        return out                            


train_dataset = TensorDataset(train_X, train_Y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(test_X, test_Y)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model = model()
criterion = nn.BCEWithLogitsLoss()              
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x).squeeze(-1)      
        loss = criterion(outputs, batch_y)        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x).squeeze(-1)    
        loss = criterion(outputs, batch_y)
        test_loss += loss.item() * batch_x.size(0)
        
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

test_loss /= len(test_loader.dataset)
accuracy = correct / total if total > 0 else 0

print(f"Test BCE Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}  ({correct}/{total})")
print(f"All up: {test_Y.mean():.4f}")

Epoch 1/3, Loss: 0.6867
Epoch 2/3, Loss: 0.6859
Epoch 3/3, Loss: 0.6861
Test BCE Loss: 0.6923
Test Accuracy: 0.5250  (525/1000)
All up: 0.5260


In [42]:
seq_len = 5

X = []
Y = []

for i in range(combined.shape[1] - seq_len):
    X.append(combined[:,i:(seq_len + i)])
    Y.append(log_returns[i + seq_len] > 0)

X = torch.tensor(X).float()
Y = torch.tensor(Y).float()

X = X.reshape((X.shape[0],
               X.shape[2],
               X.shape[1]))

test_size = 1000

test_X = X[X.shape[0]- test_size:, :, :]
test_Y = Y[Y.shape[0] - test_size:]


train_X = X[:-test_size, :, :]
train_Y = Y[:-test_size]


class model(nn.Module):
    def __init__(self, input_size=2, hidden_size=100, output_size=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = x.transpose(0, 1)                    
        out, _ = self.rnn(x)
        out = self.fc(out[-1])                  
        return out                            


train_dataset = TensorDataset(train_X, train_Y)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

test_dataset = TensorDataset(test_X, test_Y)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model = model()
criterion = nn.BCEWithLogitsLoss()              
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x).squeeze(-1)      
        loss = criterion(outputs, batch_y)        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")


model.eval()
test_loss = 0
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x).squeeze(-1)    
        loss = criterion(outputs, batch_y)
        test_loss += loss.item() * batch_x.size(0)
        
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

test_loss /= len(test_loader.dataset)
accuracy = correct / total if total > 0 else 0

print(f"Test BCE Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}  ({correct}/{total})")
print(f"All up: {test_Y.mean():.4f}")


Epoch 1/5, Loss: 0.6876
Epoch 2/5, Loss: 0.6874
Epoch 3/5, Loss: 0.6871
Epoch 4/5, Loss: 0.6869
Epoch 5/5, Loss: 0.6868
Test BCE Loss: 0.6918
Test Accuracy: 0.5350  (535/1000)
All up: 0.5260
